# 📚 Time Series
### 시계열

> **Section 10 of 11** · Pandas Complete Reference Guide for a JS/TS developer transitioning into BA  
> 전체 11개 섹션 중 **10번째** · JS/TS 개발자 출신 BA를 위한 Pandas 완전 참조 가이드

---
# 🎯 Learning Objective
Today I want to learn: / 오늘 배울 내용:
- [x] How to convert text to real dates with `to_datetime()` and pull out year/month/weekday/quarter with the `.dt` accessor  
`to_datetime()`으로 텍스트를 진짜 날짜로 변환하고 `.dt` accessor로 연/월/요일/분기를 꺼내는 방법
- [x] How to filter by date range using a `DatetimeIndex`, and aggregate by period with `resample()`  
`DatetimeIndex`로 날짜 범위를 필터링하고 `resample()`로 기간별 집계하는 방법
- [x] How `rolling()` computes a moving average over a fixed window, and how it differs from Section 7's `expanding()`  
`rolling()`이 고정 윈도우에 대한 이동평균을 계산하는 방법과, 7번 섹션의 `expanding()`과 다른 점

---
# 🧠 Concept

## What is it?
*(Explain it in your own words.)*

**English**
Time series tools handle the one thing every other pandas tool treats as "just another column" — that dates have an inherent **order**, and "30 days ago" or "last month" are meaningful relationships a plain string or category column doesn't have. `to_datetime()` converts text into a real date type, and everything else in this notebook builds on having that type.

**한글**
시계열 도구는 다른 모든 pandas 도구가 "그냥 또 다른 열"로 취급하는 한 가지를 특별하게 다룹니다 — 날짜에는 본질적인 **순서**가 있고, "30일 전"이나 "지난달"은 순수 문자열이나 카테고리 열에는 없는 의미 있는 관계라는 점입니다. `to_datetime()`은 텍스트를 진짜 날짜 타입으로 변환하며, 이 노트북의 나머지 모든 것은 그 타입을 가지고 있다는 전제 위에 세워집니다.

## Why do we use it?
*(When is it useful?)*

**English**
A plain string column sorts alphabetically (`"2024-1-5"` comes after `"2024-10-2"`), can't answer "what happened 7 days before this," and can't be grouped into "this week" or "this month" without a lot of manual string parsing. A real datetime column answers all of these instantly, and unlocks `resample()` and `rolling()` — tools with no real equivalent for non-time data.

**한글**
순수 문자열 열은 알파벳순으로 정렬되고(`"2024-1-5"`가 `"2024-10-2"` 뒤에 옴), "이 날짜 7일 전에 무슨 일이 있었나"에 답할 수 없으며, 수많은 수동 문자열 파싱 없이는 "이번 주"나 "이번 달"로 그룹화할 수도 없습니다. 진짜 datetime 열은 이 모든 것에 즉시 답하며, `resample()`과 `rolling()`을 가능하게 합니다 — 시간이 아닌 데이터에는 진짜 대응물이 없는 도구들입니다.

## When is it used in Business Analytics?
*(Real-world use case)*

**English**
Almost every recurring business report is fundamentally a time series question — "daily sales, but smoothed," "this month vs last month," "which weekday performs best." Section 11 builds MoM/YoY calculations directly on top of what's learned here.

**한글**
거의 모든 반복되는 비즈니스 보고서는 근본적으로 시계열 질문입니다 — "일별 매출인데 완만하게", "이번 달 vs 지난달", "어느 요일이 가장 잘 나가는지". 11번 섹션은 여기서 배운 것 바로 위에 MoM/YoY 계산을 세웁니다.

### Quick Comparison: JS/TS vs pandas / 빠른 비교

| Concept / 개념 | JavaScript / TypeScript | pandas |
|---|---|---|
| Parse a date string / 날짜 문자열 파싱 | `new Date(str)` | `pd.to_datetime(str)` |
| Get the year/month/weekday / 연/월/요일 꺼내기 | `date.getFullYear()` / `.getMonth()` / `.getDay()` | `.dt.year` / `.dt.month` / `.dt.dayofweek` |
| Filter a date range / 날짜 범위 필터링 | manual `>=` / `<=` comparison / 직접 `>=` / `<=` 비교 | `df.loc["2024-01":"2024-03"]` |
| Group into weekly/monthly totals / 주별·월별 합계로 그룹화 | manual bucketing loop / 직접 버킷 반복문 | `df.resample("ME").sum()` |
| Moving average / 이동평균 | manual sliding-window loop / 직접 슬라이딩 윈도우 반복문 | `df.rolling(7).mean()` |

---
# 📝 Syntax

## Basic Syntax

In [1]:
import pandas as pd

orders = pd.DataFrame({
    "date": ["2024-01-15", "2024-02-28", "2024-06-21"],
    "sales": [320000, 85000, 210000],
})

# Convert text to a real date type / 텍스트를 진짜 날짜 타입으로 변환
orders["date"] = pd.to_datetime(orders["date"])
print(orders.dtypes)
print()

# .dt accessor -- pull out date components / .dt accessor -- 날짜 구성 요소 꺼내기
orders["month"] = orders["date"].dt.month
print(orders)

date     datetime64[us]
sales             int64
dtype: object

        date   sales  month
0 2024-01-15  320000      1
1 2024-02-28   85000      2
2 2024-06-21  210000      6


## Common Variations

In [2]:
import pandas as pd
import numpy as np

np.random.seed(42)
daily = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=10, freq="D"),
    "sales": np.random.randint(50000, 200000, 10),
})
daily = daily.set_index("date")

# resample -- aggregate into a coarser period / resample -- 더 큰 기간 단위로 집계
print(daily.resample("W-MON")["sales"].sum())
print()

# rolling -- a moving average over a fixed window / rolling -- 고정 윈도우에 대한 이동평균
daily["ma3"] = daily["sales"].rolling(3, min_periods=1).mean().round(0)
print(daily)

date
2024-01-01     171958
2024-01-08    1154863
2024-01-15     300225
Freq: W-MON, Name: sales, dtype: int64

             sales       ma3
date                        
2024-01-01  171958  171958.0
2024-01-02  196867  184412.0
2024-01-03  181932  183586.0
2024-01-04  153694  177498.0
2024-01-05  169879  168502.0
2024-01-06  160268  161280.0
2024-01-07  104886  145011.0
2024-01-08  187337  150830.0
2024-01-09  137498  143240.0
2024-01-10  162727  162521.0


---
# 🧪 Small Examples

## Example 1 — to_datetime + dt Accessor
*(Covers source section 10-1)*

**English:** `pd.to_datetime()` converts a text column into a real date type (`datetime64`) — check `.dtype` to confirm the conversion actually happened. Once converted, the `.dt` accessor unlocks `.year`, `.month`, `.day`, `.dayofweek`, `.day_name()`, `.quarter`, and comparisons like `.dayofweek >= 5` for weekends.  
**한글:** `pd.to_datetime()`은 텍스트 열을 진짜 날짜 타입(`datetime64`)으로 변환합니다 — 변환이 실제로 일어났는지 `.dtype`으로 확인하세요. 변환한 뒤에는 `.dt` accessor로 `.year`, `.month`, `.day`, `.dayofweek`, `.day_name()`, `.quarter`, 그리고 주말을 위한 `.dayofweek >= 5` 같은 비교를 사용할 수 있습니다.

In [9]:
import pandas as pd

orders = pd.DataFrame({
    "order_id": [1, 2, 3, 4, 5],
    "date": ["2024-01-15", "2024-02-28", "2024-06-22", "2024-09-03", "2024-12-25"],
    "sales": [320000, 85000, 210000, 95000, 430000],
})

orders["date"] = pd.to_datetime(orders["date"])   # string -> datetime64 / 문자열 -> datetime64
print("dtype after conversion:", orders["date"].dtype)
print()

orders["year"] = orders["date"].dt.year
orders["month"] = orders["date"].dt.month
orders["weekday"] = orders["date"].dt.dayofweek       # 0=Monday ... 6=Sunday / 0=월요일 ... 6=일요일
orders["weekday_name"] = orders["date"].dt.day_name()
orders["quarter"] = orders["date"].dt.quarter
orders["is_weekend"] = orders["date"].dt.dayofweek >= 5

print(orders[["date", "year", "month", "weekday", "weekday_name", "quarter", "is_weekend"]])

dtype after conversion: datetime64[us]

        date  year  month  weekday weekday_name  quarter  is_weekend
0 2024-01-15  2024      1        0       Monday        1       False
1 2024-02-28  2024      2        2    Wednesday        1       False
2 2024-06-22  2024      6        5     Saturday        2        True
3 2024-09-03  2024      9        1      Tuesday        3       False
4 2024-12-25  2024     12        2    Wednesday        4       False


## Example 2 — Date Index & Period Filtering
*(Covers source section 10-2)*

**English:** Once a date column becomes the **index** (`set_index()`), slicing by a date string becomes natural: `.loc["2024-03"]` grabs every row in March, `.loc["2024-01":"2024-03"]` grabs a range. This only works through `.loc[]` on a DataFrame — `df["2024-03"]` looks for a **column** named that and raises `KeyError` (though it works directly on a Series without `.loc`).  
**한글:** 날짜 열이 **인덱스**가 되면(`set_index()`), 날짜 문자열로 슬라이싱하는 게 자연스러워집니다: `.loc["2024-03"]`은 3월의 모든 행을 가져오고, `.loc["2024-01":"2024-03"]`은 범위를 가져옵니다. 이는 DataFrame에서는 `.loc[]`을 통해서만 동작합니다 — `df["2024-03"]`은 그 이름의 **열**을 찾아서 `KeyError`를 일으킵니다(단, Series에서는 `.loc` 없이 바로 동작함).

In [4]:
import pandas as pd
import numpy as np

np.random.seed(1)
daily = pd.DataFrame({
    "date": pd.date_range("2024-01-01", "2024-12-31", freq="D"),
    "sales": np.random.randint(50000, 300000, 366),
})
daily = daily.set_index("date")   # <- required for date-string slicing / <- 날짜 문자열 슬라이싱에 필요

print("loc['2024-03'] -- every row in March:")
print(daily.loc["2024-03"].head(3))
print(f"({len(daily.loc['2024-03'])} rows total)")
print()

print("loc['2024-01':'2024-03'] -- a range of months:")
print(daily.loc["2024-01":"2024-03"].shape)
print()

# The DataFrame vs Series bracket-access gotcha / DataFrame vs Series 대괄호 접근 차이
try:
    daily["2024-03"]
except KeyError as e:
    print("daily['2024-03'] fails -- pandas looks for a COLUMN named '2024-03':", type(e).__name__)

print("but daily['sales']['2024-03'] works fine -- that's a Series, not a DataFrame:")
print(len(daily["sales"]["2024-03"]), "rows")

loc['2024-03'] -- every row in March:
             sales
date              
2024-03-01  149782
2024-03-02   66277
2024-03-03  242049
(31 rows total)

loc['2024-01':'2024-03'] -- a range of months:
(91, 1)

daily['2024-03'] fails -- pandas looks for a COLUMN named '2024-03': KeyError
but daily['sales']['2024-03'] works fine -- that's a Series, not a DataFrame:
31 rows


## Example 3 — resample: Period Aggregation
*(Covers source section 10-3)*

**English:** `resample(freq)` groups a `DatetimeIndex` into coarser periods and aggregates — conceptually a `groupby()` specialized for time. It requires a `DatetimeIndex` first (the same `set_index()` step from Example 2).  
**한글:** `resample(freq)`는 `DatetimeIndex`를 더 큰 기간 단위로 그룹화하고 집계합니다 — 개념적으로는 시간에 특화된 `groupby()`입니다. 먼저 `DatetimeIndex`가 필요합니다(Example 2와 같은 `set_index()` 단계).

### Common `freq` Codes / 자주 쓰는 freq 코드

| `freq` | Meaning / 의미 |
|---|---|
| `"D"` | daily / 일별 |
| `"W-MON"` | weekly, weeks starting Monday / 주별 (월요일 시작) |
| `"ME"` | month-end / 월말 |
| `"QE"` | quarter-end / 분기말 |
| `"YE"` | year-end / 연말 |

In [5]:
import pandas as pd
import numpy as np

np.random.seed(1)
daily = pd.DataFrame({
    "date": pd.date_range("2024-01-01", "2024-12-31", freq="D"),
    "sales": np.random.randint(50000, 300000, 366),
}).set_index("date")

# Monthly sum / mean / count, all at once / 월별 합계 / 평균 / 건수를 한 번에
monthly = daily.resample("ME")["sales"].agg(["sum", "mean", "count"])
monthly.index = monthly.index.strftime("%Y-%m")
print("resample('ME') -- monthly:")
print(monthly.head(6))
print()

print("resample('QE') -- quarterly totals:")
print(daily.resample("QE")["sales"].sum())
print()

print("resample('W-MON') -- weekly totals (first 4 weeks):")
print(daily.resample("W-MON")["sales"].sum().head(4))

resample('ME') -- monthly:
             sum           mean  count
date                                  
2024-01  5678197  183167.645161     31
2024-02  5187780  178888.965517     29
2024-03  5559899  179351.580645     31
2024-04  5766418  192213.933333     30
2024-05  5430749  175185.451613     31
2024-06  5324779  177492.633333     30

resample('QE') -- quarterly totals:
date
2024-03-31    16425876
2024-06-30    16521946
2024-09-30    16009380
2024-12-31    16155886
Freq: QE-DEC, Name: sales, dtype: int64

resample('W-MON') -- weekly totals (first 4 weeks):
date
2024-01-01     178037
2024-01-08    1255367
2024-01-15    1364048
2024-01-22    1193876
Freq: W-MON, Name: sales, dtype: int64


## Example 4 — rolling: Moving Averages
*(Covers source section 10-4)*

**English:** `.rolling(n).mean()` computes a moving average over a **fixed window** of exactly the last `n` rows — the first `n-1` rows have no full window yet, so they're `NaN` unless `min_periods=1` allows a partial one. This is different from Section 7's `.expanding()`, which accumulates from the very first row instead of a fixed-size window.  
**한글:** `.rolling(n).mean()`은 정확히 마지막 `n`개 행의 **고정 윈도우**에 대한 이동평균을 계산합니다 — 처음 `n-1`개 행은 아직 윈도우가 다 차지 않아서, `min_periods=1`로 부분 윈도우를 허용하지 않는 한 `NaN`이 됩니다. 이는 7번 섹션의 `.expanding()`과 다른데, `expanding()`은 고정 크기 윈도우 대신 첫 행부터 누적합니다.

In [ ]:
import pandas as pd

monthly = pd.DataFrame({
    "month": ["2024-01","2024-02","2024-03","2024-04","2024-05","2024-06",
              "2024-07","2024-08","2024-09","2024-10","2024-11","2024-12"],
    "sales": [5678000,5188000,5560000,5766000,5431000,5325000,
              5554000,5073000,5383000,6464000,4725000,4967000],
})

monthly["ma3"] = monthly["sales"].rolling(3).mean().round(0)
monthly["ma3_min_periods"] = monthly["sales"].rolling(3, min_periods=1).mean().round(0)
monthly["expanding_mean"] = monthly["sales"].expanding().mean().round(0)

print(monthly[["month", "sales", "ma3", "ma3_min_periods", "expanding_mean"]])
print()
print("-> ma3 has NaN for the first 2 rows (no full 3-month window yet)")
print("-> ma3_min_periods has no NaN, but months 1-2 are averages of fewer than 3 values")
print("-> expanding_mean never has NaN -- it's a running average of everything so far, not a fixed window")
print("-> ma3에는 처음 2행이 NaN(3개월 윈도우가 아직 안 참)")
print("-> ma3_min_periods는 NaN이 없지만, 1~2번째 달은 3개 미만 값의 평균")
print("-> expanding_mean은 NaN이 절대 없음 -- 고정 윈도우가 아니라 지금까지 전체의 누적 평균")

      month    sales        ma3  ma3_min_periods  expanding_mean
0   2024-01  5678000        NaN        5678000.0       5678000.0
1   2024-02  5188000        NaN        5433000.0       5433000.0
2   2024-03  5560000  5475333.0        5475333.0       5475333.0
3   2024-04  5766000  5504667.0        5504667.0       5548000.0
4   2024-05  5431000  5585667.0        5585667.0       5524600.0
5   2024-06  5325000  5507333.0        5507333.0       5491333.0
6   2024-07  5554000  5436667.0        5436667.0       5500286.0
7   2024-08  5073000  5317333.0        5317333.0       5446875.0
8   2024-09  5383000  5336667.0        5336667.0       5439778.0
9   2024-10  6464000  5640000.0        5640000.0       5542200.0
10  2024-11  4725000  5524000.0        5524000.0       5467909.0
11  2024-12  4967000  5385333.0        5385333.0       5426167.0

-> ma3 has NaN for the first 2 rows (no full 3-month window yet)
-> ma3_min_periods has no NaN, but months 1-2 are averages of fewer than 3 values
-> expa

## Example 5 — Common Combo Patterns
*(Covers source section 10-5)*

**English:** Pattern A combines the `.dt` accessor with `groupby()` to answer "which weekday performs best" — extract the weekday name, then group and average by it. Pattern B chains `resample()` + `assign()` into a single monthly report: total sales, month-over-month % change (via `pct_change()`), and a smoothed rolling average, all in one pipeline.  
**한글:** 패턴 A는 `.dt` accessor와 `groupby()`를 결합해서 "어느 요일이 가장 잘 나가는지"에 답합니다 — 요일 이름을 추출한 뒤, 그것으로 그룹화하고 평균을 냅니다. 패턴 B는 `resample()` + `assign()`을 하나의 월간 보고서로 체이닝합니다: 총매출, 전월 대비 % 변화(`pct_change()`), 그리고 완만해진 이동평균 — 전부 하나의 파이프라인으로.

In [13]:
import pandas as pd
import numpy as np

# Pattern A: dt accessor + groupby -- average sales by day of week
# 패턴 A: dt accessor + groupby -- 요일별 평균 매출
np.random.seed(3)
daily = pd.DataFrame({
    "date": pd.date_range("2024-01-01", periods=91, freq="D"),
    "sales": np.random.randint(50000, 300000, 91),
})
daily["weekday"] = daily["date"].dt.day_name()
day_order = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
weekday_avg = daily.groupby("weekday")["sales"].mean().round(0).reindex(day_order)
print("Pattern A -- average sales by weekday:")
print(weekday_avg)
print()

# Pattern B: resample + assign -- one monthly report with MoM% and a rolling average
# 패턴 B: resample + assign -- MoM%와 이동평균이 담긴 월간 보고서 한 번에
np.random.seed(1)
daily2 = pd.DataFrame({
    "date": pd.date_range("2024-01-01", "2024-06-30", freq="D"),
    "sales": np.random.randint(100000, 400000, 182),
}).set_index("date")

monthly_report = (
    daily2.resample("ME")["sales"]
    .sum()
    .reset_index()
    .assign(
        MoM_pct = lambda x: (x["sales"].pct_change() * 100).round(1),
        rolling_avg = lambda x: x["sales"].rolling(3, min_periods=1).mean().round(0),
        month = lambda x: x["date"].dt.strftime("%Y-%m"),
    )
    [["month", "sales", "MoM_pct", "rolling_avg"]]
)
print("Pattern B -- monthly report:")
print(monthly_report)

Pattern A -- average sales by weekday:
weekday
Monday       187998.0
Tuesday      183337.0
Wednesday    167581.0
Thursday     148548.0
Friday       161439.0
Saturday     187517.0
Sunday       167954.0
Name: sales, dtype: float64

Pattern B -- monthly report:
     month    sales  MoM_pct  rolling_avg
0  2024-01  8268975      NaN    8268975.0
1  2024-02  6633640    -19.8    7451308.0
2  2024-03  7176466      8.2    7359694.0
3  2024-04  8904539     24.1    7571548.0
4  2024-05  7219919    -18.9    7766975.0
5  2024-06  7372020      2.1    7832159.0


## Example 6 (Practice) — Fill in the Blanks
*(Practice built from the to_datetime + resample + rolling combo -- the source PDF has no separate numbered practice problem for this section)*

**English:** Fill in each `________` blank below. The code is syntactically valid Python, so it won't raise a `SyntaxError` — but it also won't print any result until every blank is correct (it will raise a runtime error instead, which is expected).
**한글:** 아래 `________` 빈칸을 채워보세요. 코드는 문법적으로 올바른 파이썬이라 `SyntaxError`는 나지 않지만, 모든 빈칸이 정확해지기 전까지는 결과가 출력되지 않습니다(대신 런타임 오류가 나는데, 이는 의도된 동작입니다).

In [14]:
import pandas as pd
import numpy as np

np.random.seed(5)
daily = pd.DataFrame({
    "date": pd.date_range("2024-01-01", "2024-03-31", freq="D").astype(str),   # arrives as text / 텍스트로 들어옴
    "sales": np.random.randint(80000, 250000, 91),
})

# 1. Convert "date" from text into a real datetime / "date"를 텍스트에서 진짜 datetime으로 변환
daily["date"] = pd.to_datetime(daily["date"])

# 2. Set "date" as the index so resample() can work / resample()이 동작하도록 "date"를 인덱스로 설정
daily = daily.set_index("date")

# 3. Resample to monthly totals / 월별 합계로 resample
monthly = daily.resample("ME")["sales"].sum()

# 4. Add a 2-month rolling average, with no leading NaN / 선행 NaN 없이 2개월 이동평균 추가
monthly_df = monthly.reset_index()
monthly_df["rolling_avg"] = monthly_df["sales"].rolling(2, min_periods=1).mean().round(0)

print(monthly_df)

        date    sales  rolling_avg
0 2024-01-31  5133642    5133642.0
1 2024-02-29  4670735    4902188.0
2 2024-03-31  4617704    4644220.0


### 💡 Hint / 힌트
`to_datetime` · `set_index` · `resample` · `rolling`

### ✅ Solution / 정답
*(Try solving it yourself first! / 먼저 스스로 풀어본 뒤에 확인하세요!)*

In [ ]:
import pandas as pd
import numpy as np

np.random.seed(5)
daily = pd.DataFrame({
    "date": pd.date_range("2024-01-01", "2024-03-31", freq="D").astype(str),
    "sales": np.random.randint(80000, 250000, 91),
})

daily["date"] = pd.to_datetime(daily["date"])
daily = daily.set_index("date")
monthly = daily.resample("ME")["sales"].sum()

monthly_df = monthly.reset_index()
monthly_df["rolling_avg"] = monthly_df["sales"].rolling(2, min_periods=1).mean().round(0)

print(monthly_df)

# min_periods=1 means January's rolling_avg is just January's own total (no prior month
# to average with) -- that's expected, not a bug, for the very first row of any rolling window.
# min_periods=1이므로 1월의 rolling_avg는 그냥 1월 자신의 합계임(평균 낼 이전 달이 없음) --
# 이는 버그가 아니라, rolling 윈도우의 가장 첫 행에서는 당연한 결과입니다.

---
# ⚠️ Common Mistakes

### Mistake 1 — Using `df["2024-03"]` instead of `df.loc["2024-03"]`
**English:** On a DataFrame with a `DatetimeIndex`, `df["2024-03"]` raises `KeyError` — bracket access on a DataFrame looks for a **column** named `"2024-03"`, which doesn't exist. Date-string slicing only works through `.loc[]`.  
**한글:** `DatetimeIndex`를 가진 DataFrame에서 `df["2024-03"]`은 `KeyError`를 일으킵니다 — DataFrame의 대괄호 접근은 `"2024-03"`이라는 이름의 **열**을 찾는데, 그런 열은 없기 때문입니다. 날짜 문자열 슬라이싱은 `.loc[]`을 통해서만 동작합니다.

**✅ Fix / 해결법:**  
Always use `.loc["2024-03"]` on a DataFrame. (On a plain Series, bracket access works directly without `.loc` — but it's safer to just always use `.loc[]` out of habit.)  
DataFrame에서는 항상 `.loc["2024-03"]`을 사용하세요. (순수 Series에서는 `.loc` 없이 대괄호 접근이 바로 동작하지만, 습관적으로 항상 `.loc[]`을 쓰는 게 더 안전합니다.)

### Mistake 2 — Calling `resample()` before setting a `DatetimeIndex`
**English:** `df.resample("ME")` fails (or behaves unexpectedly) if `df`'s index is still the default `RangeIndex` — having a datetime-typed **column** isn't enough; `resample()` specifically needs a `DatetimeIndex`.  
**한글:** `df`의 인덱스가 여전히 기본 `RangeIndex`라면 `df.resample("ME")`는 실패하거나(또는 예상치 못하게 동작합니다) — datetime 타입의 **열**이 있는 것만으로는 충분하지 않으며, `resample()`은 특별히 `DatetimeIndex`를 필요로 합니다.

**✅ Fix / 해결법:**  
Always `df = df.set_index("date_col")` (after `to_datetime()`) before calling `resample()`.  
`resample()`을 호출하기 전에 항상 (`to_datetime()` 이후) `df = df.set_index("date_col")`을 실행하세요.

### Mistake 3 — Confusing `rolling(n)` with `expanding()`
**English:** `.rolling(3).mean()` always averages exactly the last 3 rows — old data eventually falls out of the window. `.expanding().mean()` averages **everything since the start** — nothing ever falls out. Using one when the report actually needed the other produces a plausible-looking but wrong trend line.  
**한글:** `.rolling(3).mean()`은 항상 정확히 마지막 3개 행을 평균 냅니다 — 오래된 데이터는 결국 윈도우에서 빠집니다. `.expanding().mean()`은 **시작부터 지금까지 전부**를 평균 냅니다 — 아무것도 빠지지 않습니다. 보고서가 실제로는 다른 쪽을 필요로 했는데 한쪽을 쓰면, 그럴듯해 보이지만 틀린 추세선이 나옵니다.

**✅ Fix / 해결법:**  
Ask "should old data eventually stop influencing this average?" — if yes, `rolling(n)`; if the average should reflect the entire history to date, `expanding()`.  
"오래된 데이터가 결국 이 평균에 영향을 안 줘야 하는가?"를 물어보세요 — 그렇다면 `rolling(n)`, 평균이 지금까지의 전체 이력을 반영해야 한다면 `expanding()`입니다.

---
# 💡 Tips
Useful tips or shortcuts / 유용한 팁과 단축법

- Run `pd.to_datetime(..., errors="coerce")` on a freshly-loaded date column before doing anything else with it — an invalid date becomes `NaT` instead of silently staying as unusable text.  
새로 불러온 날짜 열에는 다른 작업을 하기 전에 항상 `pd.to_datetime(..., errors="coerce")`를 실행하세요 — 잘못된 날짜가 조용히 쓸모없는 텍스트로 남는 대신 `NaT`가 됩니다.
- Set `min_periods=1` on `rolling()` whenever a report shouldn't have blank cells for the first few rows — but weigh that against whether an average of only 1-2 data points is actually meaningful for your use case.  
보고서의 처음 몇 행이 빈 셀이면 안 될 때는 `rolling()`에 `min_periods=1`을 설정하세요 — 다만 데이터 1~2개만의 평균이 실제로 의미가 있는지는 따져보세요.
- `freq="ME"` (month-end) is usually what "monthly" means in a business report — `freq="MS"` (month-start) exists too, so double-check which one your `resample()` actually needs.  
비즈니스 보고서에서 "월별"은 보통 `freq="ME"`(월말)를 의미합니다 — `freq="MS"`(월초)도 있으니, `resample()`에 실제로 어느 것이 필요한지 다시 확인하세요.
- `.dt.day_name()` gives a readable weekday name directly — no need for a manual `0=Monday, 1=Tuesday` lookup table.  
`.dt.day_name()`은 읽기 쉬운 요일 이름을 바로 줍니다 — `0=월요일, 1=화요일` 같은 수동 조회표가 필요 없습니다.

---
# 🔗 Related Concepts

```
Aggregation & GroupBy    (Section 7 -- resample() IS a groupby, specialized for time)
    ↓
Time Series                 ← you are here / 지금 여기 (Section 10)
    ↓
BA Techniques              (Section 11 -- MoM/YoY calculations are shift() applied to exactly what you built here)
```

*How is today's topic connected to other concepts?*

**English:** `resample()` is conceptually `groupby(pd.Grouper(freq=...))` from Section 7, just with a friendlier name — the same split-apply-combine idea, specialized for time. `rolling()` and `expanding()` both extend Section 7's `shift()` idea (looking at neighboring rows) into a whole window instead of a single offset. Section 11's MoM% / YoY% calculations are literally `shift()` + `pct_change()` run on a `resample()` result — everything in Section 11 assumes today's tools are already second nature.

**한글:** `resample()`은 개념적으로 7번 섹션의 `groupby(pd.Grouper(freq=...))`에 더 친숙한 이름을 붙인 것입니다 — 같은 분할-적용-결합 아이디어를 시간에 특화시킨 것입니다. `rolling()`과 `expanding()`은 둘 다 7번 섹션의 `shift()` 아이디어(이웃 행을 보는 것)를 단일 오프셋이 아니라 윈도우 전체로 확장한 것입니다. 11번 섹션의 MoM% / YoY% 계산은 말 그대로 `resample()` 결과에 `shift()` + `pct_change()`를 실행한 것입니다 — 11번 섹션의 모든 내용은 오늘 배운 도구가 이미 익숙하다는 전제로 진행됩니다.

---
# 💼 Business Example
*How would a Business Analyst use this?*

**Scenario / 시나리오**

**English:** You've tracked daily sales for a full quarter. Roll it up into weekly totals, smooth out day-to-day noise with a 3-week rolling average, and report which single week performed best.

**한글:** 한 분기 전체의 일별 매출을 추적했습니다. 이를 주별 합계로 집계하고, 3주 이동평균으로 일별 노이즈를 완만하게 만든 뒤, 가장 실적이 좋았던 단일 주를 보고하세요.

**To Do / 할 일**
- [x] Convert the date column and set it as the index  
날짜 열을 변환하고 인덱스로 설정하기
- [x] Roll daily data up into weekly totals with `resample()`  
`resample()`로 일별 데이터를 주별 합계로 집계하기
- [x] Add a 3-week rolling average  
3주 이동평균 추가하기
- [x] Find the single best week  
가장 실적이 좋았던 단일 주 찾기

In [15]:
import pandas as pd
import numpy as np

np.random.seed(7)
daily_sales = pd.DataFrame({
    "date": pd.date_range("2024-04-01", "2024-06-30", freq="D").astype(str),   # arrives as text / 텍스트로 들어옴
    "sales": np.random.randint(150000, 450000, 91),
})

# 1) Convert + set index / 변환 + 인덱스 설정
daily_sales["date"] = pd.to_datetime(daily_sales["date"])
daily_sales = daily_sales.set_index("date")

# 2) Weekly totals / 주별 합계
weekly = daily_sales.resample("W-MON")["sales"].sum().reset_index()

# 3) 3-week rolling average to smooth the trend / 3주 이동평균으로 추세 완만화
weekly["rolling_3wk"] = weekly["sales"].rolling(3, min_periods=1).mean().round(0)
print(weekly)
print()

# 4) The single best week / 가장 실적이 좋았던 단일 주
best_week = weekly.loc[weekly["sales"].idxmax()]
print("Best single week:")
print(best_week)

         date    sales  rolling_3wk
0  2024-04-01   211615     211615.0
1  2024-04-08  1928532    1070074.0
2  2024-04-15  1991263    1377137.0
3  2024-04-22  1937670    1952488.0
4  2024-04-29  2064137    1997690.0
5  2024-05-06  1933863    1978557.0
6  2024-05-13  2078832    2025611.0
7  2024-05-20  1847691    1953462.0
8  2024-05-27  1897853    1941459.0
9  2024-06-03  2175510    1973685.0
10 2024-06-10  2503008    2192124.0
11 2024-06-17  2024766    2234428.0
12 2024-06-24  1762593    2096789.0
13 2024-07-01  1863194    1883518.0

Best single week:
date           2024-06-10 00:00:00
sales                      2503008
rolling_3wk              2192124.0
Name: 10, dtype: object


---
# 📝 Summary
*Write today's concept in 3~5 sentences.*

**English**
Time series tools all depend on one foundation: a real datetime type, built with `pd.to_datetime()`. Once a column (or better, the index, via `set_index()`) holds real dates, the `.dt` accessor extracts components like year, month, weekday name, and quarter; a `DatetimeIndex` supports slicing by date string (`.loc["2024-03"]`, `.loc["2024-01":"2024-03"]`) far more naturally than manual comparisons; `resample()` regroups daily data into weekly, monthly, or quarterly totals (conceptually a `groupby()` specialized for time); and `rolling(n)` computes a moving statistic over a fixed window of the last `n` rows — smoothing out noise — while Section 7's `expanding()` instead accumulates from the very first row. Combined, `.dt` + `groupby()` reveals patterns like "which weekday performs best," and `resample()` + `rolling()` builds the classic "smoothed trend line" report.

**한글**
시계열 도구는 모두 하나의 기반에 의존합니다: `pd.to_datetime()`으로 만든 진짜 datetime 타입. 열(또는 더 낫게는, `set_index()`를 통한 인덱스)이 진짜 날짜를 담고 있으면, `.dt` accessor는 연도, 월, 요일 이름, 분기 같은 구성 요소를 추출하고, `DatetimeIndex`는 직접 비교보다 훨씬 자연스럽게 날짜 문자열로 슬라이싱을 지원합니다(`.loc["2024-03"]`, `.loc["2024-01":"2024-03"]`). `resample()`은 일별 데이터를 주별, 월별, 분기별 합계로 재그룹화하고(개념적으로는 시간에 특화된 `groupby()`), `rolling(n)`은 마지막 `n`개 행의 고정 윈도우에 대한 이동 통계를 계산해서 노이즈를 완만하게 만드는 반면, 7번 섹션의 `expanding()`은 첫 행부터 누적합니다. 이를 합치면, `.dt` + `groupby()`는 "어느 요일이 가장 잘 나가는지" 같은 패턴을 드러내고, `resample()` + `rolling()`은 전형적인 "완만해진 추세선" 보고서를 만듭니다.

---
# 📌 One Sentence Summary
Today's topic in ONE sentence.

> Once a column becomes a real datetime — via `to_datetime()` — everything else follows: extracting date parts with `.dt`, slicing by date range with a `DatetimeIndex`, regrouping periods with `resample()`, and smoothing noise with `rolling()`.

> 열이 `to_datetime()`을 통해 진짜 datetime이 되고 나면, 나머지는 자연스럽게 따라옵니다: `.dt`로 날짜 구성 요소 추출하기, `DatetimeIndex`로 날짜 범위 슬라이싱하기, `resample()`로 기간 재그룹화하기, `rolling()`으로 노이즈 완만하게 만들기.

---
# ❓ Review Questions

**Q1.** Why does `orders["date"].dt.year` work but `orders.dt.year` (without `["date"]`) doesn't?
**Q1.** `orders["date"].dt.year`는 동작하는데 `orders.dt.year`(`["date"]` 없이)는 왜 안 되나요?

orders["date"].dt.year works because the date column is a datetime Series. orders.dt.year does not work because orders itself is a DataFrame.  
orders["date"].dt.year는 date라는 Series가 datetime 타입이기 때문에 .dt를 사용할 수 있음. orders.dt.year는 orders 자체가 DataFrame이므로 .dt를 사용할 수 없음.

**Q2.** Why does `daily["2024-03"]` raise a `KeyError` when `daily.loc["2024-03"]` works fine?
**Q2.** `daily.loc["2024-03"]`은 잘 되는데 `daily["2024-03"]`은 왜 `KeyError`를 일으키나요?

daily["2024-03"] looks for a column named "2024-03", while daily.loc["2024-03"] looks for an index label named "2024-03".  
daily["2024-03"]은 column을 찾는 문법이고, daily.loc["2024-03"]은 index label을 찾는 문법.

**Q3.** What does `daily.resample("ME")` need to already be true about `daily` before it will work?
**Q3.** `daily.resample("ME")`가 동작하려면 `daily`에 대해 미리 무엇이 참이어야 하나요?

Before daily.resample("ME") can work, daily must have a DatetimeIndex.  
daily.resample("ME")가 동작하려면 daily의 index가 DatetimeIndex여야함.

**Q4.** What's the key difference between `.rolling(3).mean()` and `.expanding().mean()`?
**Q4.** `.rolling(3).mean()`과 `.expanding().mean()`의 핵심 차이는 무엇인가요?

rolling(3).mean() calculates a moving average using the most recent 3 values, while expanding().mean() calculates a cumulative average from the beginning up to the current row.  
rolling(3).mean()은 최근 3개 값만 사용하는 이동 평균이고, expanding().mean()은 처음부터 현재 행까지 모든 값을 누적해서 평균을 계산.

**Q5.** Why does `.rolling(3).mean()` produce `NaN` for the first two rows, and how do you avoid that?
**Q5.** `.rolling(3).mean()`은 왜 처음 두 행에 대해 `NaN`을 만들며, 이를 어떻게 피할 수 있나요?

rolling(3).mean() produces NaN for the first two rows because three values are required by default. You can avoid this by setting min_periods.
rolling(3).mean()은 평균을 계산하려면 기본적으로 3개의 값이 필요하기 때문에 처음 2개에는 NaN이 생김. 이를 피하려면 min_periods를 지정.  

---
*📅 Try answering these again in a few days. / 며칠 뒤에 다시 답해보세요.*